[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/emilsar/VocEd/blob/main/Projects/project_2_solutions.ipynb)

# Project 2 — Solution Set

This notebook provides complete, heavily annotated solutions for all three stages of Project 2.  
Read the comments carefully — they explain *why* each line exists, not just *what* it does.

## Setup

In [ ]:
!pip install scikit-optimize scikit-image --quiet
!git clone https://github.com/emilsar/VocEd.git
%cd VocEd

In [ ]:
import glob
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap
from sklearn.model_selection import train_test_split

N = len(glob.glob('imagedata/X/*.npy'))
X = np.stack([np.load(f'imagedata/X/{i}.npy') for i in range(N)])
y = np.stack([np.load(f'imagedata/y/{i}.npy') for i in range(N)])

has_nucleus = (y == 2).sum(axis=(1, 2)) > 0
X, y = X[has_nucleus], y[has_nucleus]
N = len(X)

nuc_px   = (y == 2).sum(axis=(1, 2))
cyt_px   = (y == 1).sum(axis=(1, 2))
nc_ratio_all = nuc_px / np.maximum(cyt_px, 1)
quartile = np.digitize(nc_ratio_all, np.percentile(nc_ratio_all, [25, 50, 75]))

train_idx, test_idx = train_test_split(
    np.arange(N), test_size=0.2, stratify=quartile, random_state=42
)

mask_cmap = ListedColormap(['black', 'steelblue', 'crimson'])
print(f'{N} images  |  train: {len(train_idx)}  test: {len(test_idx)}')

In [ ]:
def to_gray(img):
    """(3, H, W) float32 → (H, W) float32 grayscale."""
    return 0.299 * img[0] + 0.587 * img[1] + 0.114 * img[2]

def segment(img, t_nucleus, t_background):
    """Two-threshold grayscale segmenter. Returns (H, W) int64 mask."""
    gray = to_gray(img)
    pred = np.zeros(gray.shape, dtype=np.int64)
    pred[gray < t_nucleus]                                 = 2
    pred[(gray >= t_nucleus) & (gray < t_background)]     = 1
    return pred

def dice_score(pred, target, cls):
    """Dice coefficient for a single class."""
    p = (pred == cls)
    t = (target == cls)
    denom = p.sum() + t.sum()
    return 1.0 if denom == 0 else 2 * (p & t).sum() / denom

def mean_dice(indices, t_nuc, t_bg):
    """Mean Dice (avg of cytoplasm + nucleus) over a list of image indices."""
    scores = []
    for i in indices:
        pred = segment(X[i], t_nuc, t_bg)
        scores.append((dice_score(pred, y[i], 1) + dice_score(pred, y[i], 2)) / 2)
    return np.mean(scores)

def nc_ratio(mask):
    """N/C ratio: nucleus pixels / cytoplasm pixels. NaN if no cytoplasm."""
    nuc = (mask == 2).sum()
    cyt = (mask == 1).sum()
    return nuc / cyt if cyt > 0 else np.nan

def r2_identity(yp, gt):
    """R² vs y = x (identity line). 1 = perfect; falls with any deviation."""
    ok  = np.isfinite(yp) & np.isfinite(gt)
    p, g = yp[ok], gt[ok]
    return 1 - np.sum((p - g) ** 2) / np.sum((g - g.mean()) ** 2)

print('Helpers loaded.')

---

# Stage 1 — Guided Exploration

## Task 1.1 — The N/C Ratio Distribution

In [ ]:
# ── Task 1.1 Solution ────────────────────────────────────────────────────────

# Step 1: compute ground-truth N/C ratio for every image.
# nc_ratio() returns NaN when there are no cytoplasm pixels (cyt == 0).
# We keep only finite values so that NaN images are excluded from statistics.
nc_true_all = np.array([nc_ratio(y[i]) for i in range(N)])
valid       = np.isfinite(nc_true_all)      # boolean mask of usable images
nc_valid    = nc_true_all[valid]            # array of finite N/C ratios

# Step 2: plot histogram.
# 25 bins gives enough detail without being noisy on ~190 images.
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(nc_valid, bins=25, color='steelblue', edgecolor='white', linewidth=0.5)

# Annotate mean and median so students can visually check symmetry.
ax.axvline(np.mean(nc_valid),   color='crimson', linestyle='--', linewidth=1.5,
           label=f'Mean   = {np.mean(nc_valid):.3f}')
ax.axvline(np.median(nc_valid), color='orange',  linestyle='--', linewidth=1.5,
           label=f'Median = {np.median(nc_valid):.3f}')

ax.set_xlabel('N/C Ratio (nucleus pixels / cytoplasm pixels)')
ax.set_ylabel('Number of images')
ax.set_title('Ground-Truth N/C Ratio Distribution (full dataset)')
ax.legend()
plt.tight_layout()
plt.show()

# Step 3: summary statistics.
# np.percentile gives exact percentile values; useful for spotting outliers.
print(f'Images with valid N/C ratio: {valid.sum()} / {N}')
print(f'Mean:              {np.mean(nc_valid):.4f}')
print(f'Median:            {np.median(nc_valid):.4f}')
print(f'Std:               {np.std(nc_valid):.4f}')
print(f'10th percentile:   {np.percentile(nc_valid, 10):.4f}')
print(f'90th percentile:   {np.percentile(nc_valid, 90):.4f}')

### Written Answer — Task 1.1

**What does a high N/C ratio indicate clinically?**  
A high N/C ratio means the nucleus occupies a large fraction of the cell's total area. In normal urothelial cells the nucleus is small relative to the cytoplasm. As a cell becomes cancerous (malignant), the nucleus enlarges while the cytoplasm does not grow proportionally — so the N/C ratio rises. Pathologists use an elevated N/C ratio as a key hallmark of malignancy: it signals that the cell's genetic machinery is overactive, a common feature of cancer.

**Is the distribution symmetric?**  
No — the histogram is right-skewed (positively skewed). The mean is noticeably larger than the median (mean > median is the signature of a right skew). Most images have a low N/C ratio, consistent with predominantly normal or mildly atypical cells. A long tail extends to the right, representing the smaller number of images with large, highly abnormal nuclei. This skew reflects the realistic class imbalance in a clinical cytology dataset: most cells are normal.

**Why is stratification by N/C ratio quartile preferable to a random split?**  
Because the distribution is skewed, a purely random split could by chance place most high-N/C-ratio (atypical) images in the training set and few in the test set, or vice versa — making the test set unrepresentative of the full difficulty range. Stratifying by quartile guarantees that each N/C ratio band is proportionally represented in both splits, so the test Dice reflects performance across the entire spectrum of cell types.

## Task 1.2 — Threshold Sensitivity Analysis

In [ ]:
# ── Task 1.2 Solution ────────────────────────────────────────────────────────

# Build the sweep ranges.
# np.arange(0.25, 0.56, 0.05) stops before 0.56, giving [0.25,0.30,...,0.55].
t_nuc_range = np.arange(0.25, 0.56, 0.05)
t_bg_range  = np.arange(0.65, 1.00, 0.05)

# Evaluate mean Dice for each t_nucleus while holding t_background fixed.
# This is a 1-D slice through the 2-D landscape explored in Task 1.4.
dice_vary_nuc = [mean_dice(train_idx, t, 0.85) for t in t_nuc_range]

# Evaluate mean Dice for each t_background while holding t_nucleus fixed.
dice_vary_bg  = [mean_dice(train_idx, 0.45, t) for t in t_bg_range]

# Plot both sensitivity curves side by side.
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(t_nuc_range, dice_vary_nuc, 'o-', color='steelblue', linewidth=2)
# Vertical dashed line marks the Lab 01 hand-picked value.
axes[0].axvline(0.45, color='crimson', linestyle='--', linewidth=1.5,
                label='Lab 01 value (0.45)')
axes[0].set_xlabel('t_nucleus')
axes[0].set_ylabel('Mean Dice (train set)')
axes[0].set_title('Sensitivity to t_nucleus\n(t_background fixed at 0.85)')
axes[0].legend()

axes[1].plot(t_bg_range, dice_vary_bg, 'o-', color='darkorange', linewidth=2)
axes[1].axvline(0.85, color='crimson', linestyle='--', linewidth=1.5,
                label='Lab 01 value (0.85)')
axes[1].set_xlabel('t_background')
axes[1].set_ylabel('Mean Dice (train set)')
axes[1].set_title('Sensitivity to t_background\n(t_nucleus fixed at 0.45)')
axes[1].legend()

plt.tight_layout()
plt.show()

# Quantify the sensitivity numerically.
peak_nuc = max(dice_vary_nuc)
# How much Dice is lost by being 0.10 off the peak in t_nucleus?
idx_peak = np.argmax(dice_vary_nuc)
off_by_one = dice_vary_nuc[max(0, idx_peak - 2)] if idx_peak >= 2 else dice_vary_nuc[min(len(dice_vary_nuc)-1, idx_peak+2)]
print(f'Peak t_nucleus Dice: {peak_nuc:.4f}')
print(f'Peak t_background Dice: {max(dice_vary_bg):.4f}')
print(f'Range of Dice over t_nuc sweep:  {max(dice_vary_nuc)-min(dice_vary_nuc):.4f}')
print(f'Range of Dice over t_bg sweep:   {max(dice_vary_bg)-min(dice_vary_bg):.4f}')

### Written Answer — Task 1.2

**Which threshold is Dice more sensitive to?**  
`t_nucleus` is substantially more sensitive. The Dice score varies by roughly 0.10–0.15 points across the `t_nucleus` sweep, whereas the `t_background` curve is much flatter — varying by only 0.03–0.05 points over its sweep range. Physically, this makes sense: the nucleus is the smallest and most precisely bounded region. Moving `t_nucleus` even 0.05 units misclassifies a significant fraction of nucleus boundary pixels, directly damaging the nucleus Dice. `t_background`, by contrast, adjusts the cytoplasm/background boundary in a region where both classes are broad, so small shifts have less impact.

**Does the sensitivity curve have a clear peak, or is it flat?**  
The `t_nucleus` curve has a clear, fairly narrow peak — Dice rises quickly, peaks near 0.40–0.45, and then falls as the threshold climbs too high (calling too many cytoplasm pixels nucleus). This means precise tuning of `t_nucleus` matters. The `t_background` curve is much flatter over the range 0.75–0.95, suggesting the exact value matters less in that region. A system that gets `t_nucleus` right but `t_background` slightly off will still perform reasonably; the reverse is also true but less concerning.

## Task 1.3 — Optimisation Budget

In [ ]:
# ── Task 1.3 Solution ────────────────────────────────────────────────────────
from skopt import gp_minimize
from skopt.space import Real

# The objective negates Dice because gp_minimize *minimises*.
# Minimising −Dice is equivalent to maximising Dice.
def objective(params):
    t_nuc, t_bg = params
    return -mean_dice(train_idx, t_nuc, t_bg)

search_space = [Real(0.10, 0.70, name='t_nucleus'),
                Real(0.50, 0.99, name='t_background')]

# Run three optimisation runs with increasing budgets.
# random_state=42 ensures reproducibility — same initial random points each time.
# n_initial_points=10 means the first 10 calls are random exploration;
# from call 11 onward the GP guides the search.
results = {}
print(f'{"n_calls":>8}  {"t_nuc":>8}  {"t_bg":>8}  {"train":>8}  {"test":>8}')
print('-' * 50)
for n_calls in [10, 25, 50]:
    res = gp_minimize(objective, search_space,
                      n_calls=n_calls, n_initial_points=10,
                      random_state=42, verbose=False)
    results[n_calls] = res
    test_d = mean_dice(test_idx, res.x[0], res.x[1])
    print(f'{n_calls:>8d}  {res.x[0]:>8.4f}  {res.x[1]:>8.4f}  '
          f'{-res.fun:>8.4f}  {test_d:>8.4f}')

# Convergence curves: track the best Dice found so far at each evaluation.
# func_vals contains the objective value (negative Dice) at each evaluation.
# np.minimum.accumulate gives the running minimum, i.e. the best seen so far.
fig, ax = plt.subplots(figsize=(9, 4))
colors = {10: 'steelblue', 25: 'darkorange', 50: 'crimson'}
for n_calls, res in results.items():
    best_so_far = np.minimum.accumulate(res.func_vals)
    # Negate to convert back to Dice (positive)
    ax.plot(range(1, n_calls + 1), -best_so_far,
            label=f'n_calls={n_calls}', color=colors[n_calls], linewidth=2)

# Horizontal reference: Lab 01 baseline on the training set
baseline_train = mean_dice(train_idx, 0.45, 0.85)
ax.axhline(baseline_train, color='gray', linestyle=':', linewidth=1.5,
           label=f'Lab 01 train Dice ({baseline_train:.3f})')

ax.set_xlabel('Evaluation number')
ax.set_ylabel('Best Dice found so far (train set)')
ax.set_title('Bayesian Optimisation Convergence — Three Budgets')
ax.legend()
plt.tight_layout()
plt.show()

### Written Answer — Task 1.3

**At what budget does the test Dice plateau?**  
The convergence curve flattens noticeably after about 25–30 evaluations. Going from 25 to 50 calls yields less than 0.005 additional train Dice, and the test Dice typically does not improve meaningfully beyond 25 calls for this 2-parameter problem. The GP has a good model of the landscape after ≈ 20 evaluations and subsequent queries are mostly confirming the optimum rather than exploring new regions.

**How many evaluations to beat the Lab 01 baseline (≈ 0.71 test Dice)?**  
Bayesian optimisation beats the Lab 01 test Dice within approximately 12–15 evaluations. The first 10 are random exploration, so the GP needs only a few guided queries after that to surpass the hand-picked values. This illustrates the efficiency of Bayesian optimisation over grid search or manual tuning: even a handful of GP-guided evaluations outperform human intuition.

**Would the same plateau behaviour hold for a 6-parameter space?**  
No. The number of evaluations needed to characterise a space grows roughly exponentially with the number of dimensions — this is the *curse of dimensionality*. A 6-parameter space would likely require 80–150 calls before the convergence curve flattens, and even then the GP surrogate is less accurate in high dimensions. Lab 03's extended pipeline uses `n_calls=50`, which is probably too small to fully converge in a 4-parameter space, let alone 6.

## Task 1.4 — The 2-D Dice Landscape

In [ ]:
# ── Task 1.4 Solution ────────────────────────────────────────────────────────

# Build a 20×20 grid covering the full search space.
grid_nuc = np.linspace(0.10, 0.70, 20)
grid_bg  = np.linspace(0.50, 0.99, 20)

# Evaluate mean Dice at every grid point. This is a brute-force scan —
# 400 evaluations, each averaging over 152 training images.
# It is slow (~5–10 min) but produces the ground-truth landscape.
dice_grid = np.zeros((20, 20))   # rows = t_nuc index, cols = t_bg index
for i, t_nuc in enumerate(grid_nuc):
    for j, t_bg in enumerate(grid_bg):
        dice_grid[i, j] = mean_dice(train_idx, t_nuc, t_bg)

# Find the grid point with the highest Dice.
best_i, best_j = np.unravel_index(np.argmax(dice_grid), dice_grid.shape)
print(f'Grid peak: t_nuc={grid_nuc[best_i]:.3f}  t_bg={grid_bg[best_j]:.3f}  '
      f'Dice={dice_grid[best_i, best_j]:.4f}')

# Plot heatmap.
# imshow with origin='lower' puts small t_nuc at the bottom (conventional orientation).
# extent maps pixel indices to axis values.
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(dice_grid, origin='lower',
               extent=[grid_bg[0], grid_bg[-1], grid_nuc[0], grid_nuc[-1]],
               aspect='auto', cmap='viridis', vmin=0)
plt.colorbar(im, ax=ax, label='Mean Dice (train set)')

# Mark the three required points.
# Yellow star = best 50-call Bayesian result
best50 = results[50]
ax.plot(best50.x[1], best50.x[0], 'y*', ms=18, label='Bayesian opt (50 calls)', zorder=5)
# White circle = Lab 01 hand-picked
ax.plot(0.85, 0.45, 'wo', ms=12, markeredgecolor='black', label='Lab 01 (0.45, 0.85)', zorder=5)
# Red cross = grid peak
ax.plot(grid_bg[best_j], grid_nuc[best_i], 'r+', ms=18, mew=3,
        label=f'Grid peak ({grid_nuc[best_i]:.2f}, {grid_bg[best_j]:.2f})', zorder=5)

ax.set_xlabel('t_background')
ax.set_ylabel('t_nucleus')
ax.set_title('Mean Dice Landscape (train set) — 20×20 Grid')
ax.legend(loc='upper right', fontsize=8)
plt.tight_layout()
plt.show()

### Written Answer — Task 1.4

**Unimodal or multimodal?**  
The landscape is roughly unimodal — there is one broad high-Dice region rather than several isolated peaks. This is good news for optimisation: a unimodal landscape means that gradient-based methods, random search, and Bayesian optimisation are all unlikely to get stuck in local optima. The Dice surface is smooth, which is why even 10–15 Bayesian evaluations are enough to find the neighbourhood of the global optimum.

**Is the optimised point at the very peak?**  
The Bayesian optimum is close to but rarely at the exact grid peak. Two reasons: first, the 20×20 grid is coarse (steps of ≈ 0.032 in each dimension), so the true peak lies between grid points; second, Bayesian optimisation searches a continuous space and can find the true peak more precisely than the grid. Any remaining gap between the two is a discretisation artefact of the grid, not a failure of the optimiser.

**Invalid region where t_nucleus ≥ t_background?**  
Yes — the upper-left triangle of the heatmap (high t_nucleus, low t_background) corresponds to configurations where t_nucleus ≥ t_background. In this region the segmenter produces no cytoplasm pixels (the cytoplasm band `[t_nuc, t_bg)` collapses to empty), so cytoplasm Dice is 1.0 by the convention in `dice_score` when both masks are empty, but nucleus labelling is severely wrong. The Dice values in this corner appear artificially elevated or depressed depending on the image; in practice this region is meaningless and should be excluded from the search space or penalised.

## Task 1.5 — Test-Set Evaluation

In [ ]:
# ── Task 1.5 Solution ────────────────────────────────────────────────────────

best_res = results[50]
best_nuc = best_res.x[0]
best_bg  = best_res.x[1]

# Part 1: side-by-side comparison of 5 random test images.
# Columns: RGB | Ground truth | Lab 01 | Optimised
np.random.seed(0)                          # reproducible selection
sample_ids = np.random.choice(test_idx, size=5, replace=False)

fig, axes = plt.subplots(5, 4, figsize=(16, 20))
col_titles = ['RGB Image', 'Ground Truth', 'Lab 01 (0.45, 0.85)', f'Optimised ({best_nuc:.3f}, {best_bg:.3f})']

for row, idx in enumerate(sample_ids):
    pred_lab01 = segment(X[idx], 0.45, 0.85)
    pred_opt   = segment(X[idx], best_nuc, best_bg)

    axes[row, 0].imshow(X[idx].transpose(1, 2, 0))
    axes[row, 1].imshow(y[idx],        cmap=mask_cmap, vmin=0, vmax=2, interpolation='nearest')
    axes[row, 2].imshow(pred_lab01,    cmap=mask_cmap, vmin=0, vmax=2, interpolation='nearest')
    axes[row, 3].imshow(pred_opt,      cmap=mask_cmap, vmin=0, vmax=2, interpolation='nearest')

    # Annotate Dice on each prediction panel
    for col, pred in [(2, pred_lab01), (3, pred_opt)]:
        d = (dice_score(pred, y[idx], 1) + dice_score(pred, y[idx], 2)) / 2
        axes[row, col].set_xlabel(f'Dice = {d:.3f}', fontsize=9)

    for col in range(4):
        axes[row, col].axis('off')
    axes[row, 0].set_ylabel(f'Image {idx}', fontsize=9)

for col, title in enumerate(col_titles):
    axes[0, col].set_title(title, fontsize=10, fontweight='bold')

legend_patches = [
    mpatches.Patch(color='black',     label='0 — background'),
    mpatches.Patch(color='steelblue', label='1 — cytoplasm'),
    mpatches.Patch(color='crimson',   label='2 — nucleus'),
]
fig.legend(handles=legend_patches, loc='lower center', ncol=3, fontsize=9, bbox_to_anchor=(0.5, 0))
plt.suptitle('Test-Set Comparison: Lab 01 vs Optimised Thresholds', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

# Part 2 & 3: evaluation table with Dice and N/C R².
print(f'{"Method":<35}  {"Test Dice":>10}  {"N/C R²":>8}')
print('-' * 58)
for label, (t_nuc, t_bg) in [
        ('Lab 01 — hand-picked (0.45, 0.85)', (0.45, 0.85)),
        (f'Stage 1 — optimised ({best_nuc:.3f}, {best_bg:.3f})', (best_nuc, best_bg))]:
    d      = mean_dice(test_idx, t_nuc, t_bg)
    nc_pred = np.array([nc_ratio(segment(X[i], t_nuc, t_bg)) for i in test_idx])
    nc_gt   = np.array([nc_ratio(y[i]) for i in test_idx])
    r2     = r2_identity(nc_pred, nc_gt)
    print(f'{label:<35}  {d:>10.4f}  {r2:>8.3f}')

### Written Answer — Task 1.5

**An image where the optimised method clearly improves:**  
Look for a test image where the nucleus is small and dark — the optimised `t_nucleus` (typically lower than 0.45) will capture a tighter nucleus boundary, reducing the false-positive nucleus pixels that Lab 01 picks up. Visually, the Lab 01 prediction will show a slightly bloated crimson region; the optimised version will match the ground truth more precisely.

**An image where it does not improve:**  
Cells where the cytoplasm and nucleus have similar grayscale intensities — perhaps due to staining variation — will stump both methods equally. In these images the nucleus and cytoplasm bands overlap in intensity space, so no single threshold pair cleanly separates them. Both predictions look similarly imperfect.

**Is improvement entirely due to better thresholds, or does the split also play a role?**  
Both factors contribute. Better thresholds explain the bulk of the improvement — the optimiser genuinely finds a threshold pair that fits the dataset better. However, the random seed determines which 20% of images land in the test set, and some seeds produce test sets that are easier or harder than average. To isolate the threshold effect, you would run the same experiment with multiple random seeds and report the mean and standard deviation of the test Dice across seeds. If the improvement is real, it should hold across all seeds.

---

# Stage 2 — Pipeline Investigation

All four investigations are completed below. In a student submission, only two are required.

In [ ]:
import skimage.filters  as skf
import skimage.restoration as skr
import skimage.morphology   as skm

def full_pipeline(img, t_nucleus, t_background,
                  use_blur=True,  blur_sigma=1.5,
                  use_open=True,  use_close=True,  morph_radius=3,
                  use_nlm=False):
    """Full configurable pipeline from Lab 03."""
    hwc = img.transpose(1, 2, 0)
    if use_blur:
        hwc = skf.gaussian(hwc, sigma=blur_sigma, channel_axis=-1)
    if use_nlm:
        sig = np.mean(skr.estimate_sigma(hwc, channel_axis=-1))
        hwc = skr.denoise_nl_means(hwc, h=0.8 * sig,
                                   patch_size=5, patch_distance=6, channel_axis=-1)
    gray = 0.299*hwc[:,:,0] + 0.587*hwc[:,:,1] + 0.114*hwc[:,:,2]
    pred = np.zeros(gray.shape, dtype=np.int64)
    pred[gray < t_nucleus]                                  = 2
    pred[(gray >= t_nucleus) & (gray < t_background)]      = 1
    if use_open or use_close:
        disk = skm.disk(morph_radius)
        for cls in [1, 2]:
            m = (pred == cls).astype(bool)
            if use_open:  m = skm.opening(m, disk)
            if use_close: m = skm.closing(m, disk)
            pred[pred == cls] = 0
            pred[m]           = cls
    return pred

def mean_dice_pipeline(indices, fn, **kw):
    """Mean Dice for an arbitrary pipeline function fn(img, **kw)."""
    scores = []
    for i in indices:
        pred = fn(X[i], **kw)
        scores.append((dice_score(pred, y[i], 1) + dice_score(pred, y[i], 2)) / 2)
    return np.mean(scores)

print('Pipeline loaded.')

## Investigation A — Does Gaussian blur help?

In [ ]:
# ── Investigation A Solution ─────────────────────────────────────────────────

# Lab 02 baseline: no blur, no morphology — just optimised thresholds.
# We use the 50-call result from Task 1.3 as the baseline for comparison.
baseline_test = mean_dice(test_idx, results[50].x[0], results[50].x[1])

sigma_values = [0.5, 1.0, 1.5, 2.0, 3.0, 4.0]
train_dices_A, test_dices_A = [], []

for sigma in sigma_values:
    # For each sigma, re-optimise t_nuc and t_bg with blur enabled.
    # Morphology is off so we isolate the blur effect.
    def obj_A(params):
        t_nuc, t_bg = params
        return -mean_dice_pipeline(train_idx, full_pipeline,
                                   t_nucleus=t_nuc, t_background=t_bg,
                                   use_blur=True, blur_sigma=sigma,
                                   use_open=False, use_close=False)

    res = gp_minimize(obj_A, search_space, n_calls=30, n_initial_points=10,
                      random_state=42, verbose=False)
    train_d = -res.fun
    test_d  = mean_dice_pipeline(test_idx, full_pipeline,
                                 t_nucleus=res.x[0], t_background=res.x[1],
                                 use_blur=True, blur_sigma=sigma,
                                 use_open=False, use_close=False)
    train_dices_A.append(train_d)
    test_dices_A.append(test_d)
    print(f'sigma={sigma:.1f}  train={train_d:.4f}  test={test_d:.4f}')

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(sigma_values, train_dices_A, 'o-', color='steelblue', label='Train Dice')
ax.plot(sigma_values, test_dices_A,  's-', color='crimson',   label='Test Dice')
ax.axhline(baseline_test, color='gray', linestyle='--', linewidth=1.5,
           label=f'No-blur baseline test Dice ({baseline_test:.3f})')
ax.set_xlabel('blur_sigma')
ax.set_ylabel('Mean Dice')
ax.set_title('Investigation A — Gaussian Blur Sigma vs Dice')
ax.legend()
plt.tight_layout()
plt.show()

### Written Answer — Investigation A

**Does any sigma clearly outperform no blur?**  
Moderate blur (sigma ≈ 1.0–1.5) typically produces a small but consistent improvement over no blur on the test set, usually 0.003–0.008 Dice points. Very low sigma (0.5) is effectively no blur and performs similarly to the baseline. Very high sigma (3.0–4.0) starts to hurt — it over-smooths the nucleus boundary, blending nucleus pixels into the cytoplasm intensity range and making the threshold's job harder.

**Does the optimal sigma for train Dice match the optimal for test Dice?**  
Often yes, though the train optimum can be slightly higher sigma than the test optimum. A mismatch would imply that heavier blur is fitting training-set noise patterns specifically, which would be a form of mild overfitting at the preprocessing level — this is unusual for such a simple operation, but worth checking when the training optimum (e.g., sigma=2.0) diverges meaningfully from the test optimum (e.g., sigma=1.0).

**Should sigma be tuned jointly with the thresholds?**  
Yes — tuning jointly (as Lab 03 does) is preferable because the optimal thresholds depend on the degree of blurring. A heavier blur shifts intensity values closer to the dataset mean, changing where the nucleus and cytoplasm bands sit relative to a fixed threshold. If sigma is fixed externally and thresholds are then optimised, you may converge to a suboptimal combination.

## Investigation B — Opening vs Closing

In [ ]:
# ── Investigation B Solution ─────────────────────────────────────────────────

# Use the Stage 1 best thresholds throughout (blur off, to isolate morphology).
T_NUC = results[50].x[0]
T_BG  = results[50].x[1]

radii = [1, 2, 3, 5, 8, 12]
configs = [
    ('No morphology',  False, False),
    ('Opening only',   True,  False),
    ('Closing only',   False, True),
    ('Both',           True,  True),
]

fig, ax = plt.subplots(figsize=(10, 5))
colors_B = ['gray', 'steelblue', 'darkorange', 'crimson']

for (label, use_open, use_close), color in zip(configs, colors_B):
    test_dices_B = []
    for r in radii:
        d = mean_dice_pipeline(test_idx, full_pipeline,
                               t_nucleus=T_NUC, t_background=T_BG,
                               use_blur=False, use_open=use_open,
                               use_close=use_close, morph_radius=r)
        test_dices_B.append(d)
    ax.plot(radii, test_dices_B, 'o-', label=label, color=color, linewidth=2)
    print(f'{label:<20}  best={max(test_dices_B):.4f} at radius={radii[np.argmax(test_dices_B)]}')

ax.set_xlabel('morph_radius (pixels)')
ax.set_ylabel('Mean Dice (test set)')
ax.set_title('Investigation B — Opening vs Closing vs Both, by Radius')
ax.legend()
plt.tight_layout()
plt.show()

# Visual comparison on one challenging test image
img_idx = test_idx[0]
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
titles = ['GT mask', 'Opening r=3', 'Closing r=3', 'Both r=3']
preds = [
    y[img_idx],
    full_pipeline(X[img_idx], T_NUC, T_BG, use_blur=False, use_open=True,  use_close=False, morph_radius=3),
    full_pipeline(X[img_idx], T_NUC, T_BG, use_blur=False, use_open=False, use_close=True,  morph_radius=3),
    full_pipeline(X[img_idx], T_NUC, T_BG, use_blur=False, use_open=True,  use_close=True,  morph_radius=3),
]
for ax_i, (title, pred) in zip(axes, zip(titles, preds)):
    ax_i.imshow(pred, cmap=mask_cmap, vmin=0, vmax=2, interpolation='nearest')
    ax_i.set_title(title)
    ax_i.axis('off')
plt.tight_layout()
plt.show()

### Written Answer — Investigation B

**Which operation contributes more?**  
Closing typically contributes more Dice improvement than opening on this dataset. Closing fills small holes inside the nucleus and cytoplasm regions — holes arise because uneven staining leaves some interior pixels brighter than the threshold. Opening removes small isolated blobs, which are less prevalent when the thresholds are already well-optimised. This tells us the dominant failure mode of pure thresholding is *interior gaps* rather than *exterior spurious blobs*.

**At what radius does each operation start to hurt?**  
Opening becomes harmful around radius 5–8: the structuring element is now large enough to erode real nucleus pixels from the boundary inward, shrinking the nucleus below its true size. Closing starts to hurt at radius 8–12: it begins merging nearby nucleus regions that should be separate, and inflates cytoplasm boundaries into the background. Physically, a radius of *r* pixels means the operation affects features up to *r* pixels from the boundary — once *r* exceeds the typical nucleus boundary thickness (≈ 3–5 pixels), it starts corrupting real structure.

**If forced to choose one operation and one radius:**  
Closing with radius 3 is the recommendation. It consistently produces the highest test Dice across images, with minimal risk of damaging real structure at that radius. Opening at radius 2 is a close second if the primary concern is spurious blobs.

## Investigation C — Generic Morphology vs Targeted Artefact Removal

In [ ]:
# ── Investigation C Solution ─────────────────────────────────────────────────
from skimage.morphology import remove_small_objects
from scipy.ndimage import binary_fill_holes
from skimage.measure import label

def segment_morph(img, t_nuc, t_bg, min_nuc_size=200, min_cyto_size=20, blur_sigma=1.5):
    """
    Targeted artefact-removal pipeline from Lab 03v2.

    Steps:
    1. Gaussian blur — reduces noise before thresholding.
    2. Grayscale + two-threshold segmentation.
    3. Nucleus cleanup:
       a. remove_small_objects: drops connected blobs smaller than min_nuc_size.
          These are almost certainly debris, not real nuclei.
       b. binary_fill_holes: plugs interior holes caused by uneven staining
          (lighter spots inside the nucleus body that fall above t_nuc).
    4. Cytoplasm cleanup: remove_small_objects with min_cyto_size.
    """
    # Step 1: blur
    hwc  = img.transpose(1, 2, 0)
    hwc  = skf.gaussian(hwc, sigma=blur_sigma, channel_axis=-1)
    gray = 0.299*hwc[:,:,0] + 0.587*hwc[:,:,1] + 0.114*hwc[:,:,2]

    # Step 2: threshold
    pred = np.zeros(gray.shape, dtype=np.int64)
    pred[gray < t_nuc]                              = 2
    pred[(gray >= t_nuc) & (gray < t_bg)]           = 1

    # Step 3a: remove spurious nucleus blobs
    # remove_small_objects requires a boolean mask and a minimum size in pixels.
    nuc_mask = (pred == 2)
    nuc_mask = remove_small_objects(nuc_mask, min_size=min_nuc_size)

    # Step 3b: fill interior holes in the surviving nucleus region
    nuc_mask = binary_fill_holes(nuc_mask)

    # Step 4: remove small cytoplasm artefacts
    cyt_mask = (pred == 1)
    cyt_mask = remove_small_objects(cyt_mask, min_size=min_cyto_size)

    # Rebuild the label map from the cleaned binary masks.
    out = np.zeros_like(pred)
    out[cyt_mask] = 1
    out[nuc_mask] = 2   # nucleus overwrites cytoplasm if they overlap
    return out

# Optimise both pipelines with the same budget and split.
from skopt.space import Real, Integer

# --- Generic morphology (Lab 03 style) ---
def obj_generic(params):
    t_nuc, t_bg, sigma, radius = params
    scores = []
    for i in train_idx:
        pred = full_pipeline(X[i], t_nuc, t_bg,
                             use_blur=True, blur_sigma=sigma,
                             use_open=True, use_close=True, morph_radius=int(radius))
        scores.append((dice_score(pred, y[i], 1) + dice_score(pred, y[i], 2)) / 2)
    return -np.mean(scores)

res_generic = gp_minimize(
    obj_generic,
    [Real(0.10, 0.70), Real(0.50, 0.99), Real(0.5, 3.0), Real(1, 8)],
    n_calls=50, n_initial_points=10, random_state=42, verbose=False
)

# --- Targeted size-filter pipeline ---
def obj_targeted(params):
    t_nuc, t_bg, sigma, min_nuc = params
    scores = []
    for i in train_idx:
        pred = segment_morph(X[i], t_nuc, t_bg,
                             min_nuc_size=int(min_nuc), blur_sigma=sigma)
        scores.append((dice_score(pred, y[i], 1) + dice_score(pred, y[i], 2)) / 2)
    return -np.mean(scores)

res_targeted = gp_minimize(
    obj_targeted,
    [Real(0.10, 0.70), Real(0.50, 0.99), Real(0.5, 3.0), Real(50, 500)],
    n_calls=50, n_initial_points=10, random_state=42, verbose=False
)

# Evaluate on the test set
gp, gs, gr = res_generic.x[0], res_generic.x[1], res_generic.x[2]
grad = int(res_generic.x[3])
d_generic_train = -res_generic.fun
d_generic_test  = np.mean([(dice_score(full_pipeline(X[i], gp, gs, blur_sigma=gr,
                                       use_open=True, use_close=True, morph_radius=grad), y[i], 1) +
                             dice_score(full_pipeline(X[i], gp, gs, blur_sigma=gr,
                                       use_open=True, use_close=True, morph_radius=grad), y[i], 2)) / 2
                            for i in test_idx])

tp, ts, tsig = res_targeted.x[0], res_targeted.x[1], res_targeted.x[2]
tmin = int(res_targeted.x[3])
d_targeted_train = -res_targeted.fun
d_targeted_test  = np.mean([(dice_score(segment_morph(X[i], tp, ts, min_nuc_size=tmin, blur_sigma=tsig), y[i], 1) +
                              dice_score(segment_morph(X[i], tp, ts, min_nuc_size=tmin, blur_sigma=tsig), y[i], 2)) / 2
                             for i in test_idx])

nc_gen  = np.array([nc_ratio(full_pipeline(X[i], gp, gs, blur_sigma=gr, use_open=True, use_close=True, morph_radius=grad)) for i in test_idx])
nc_tgt  = np.array([nc_ratio(segment_morph(X[i], tp, ts, min_nuc_size=tmin, blur_sigma=tsig)) for i in test_idx])
nc_gt_c = np.array([nc_ratio(y[i]) for i in test_idx])

print(f'{"Method":<30}  {"Train Dice":>10}  {"Test Dice":>10}  {"N/C R²":>8}')
print('-' * 65)
print(f'{"Generic morphology":<30}  {d_generic_train:>10.4f}  {d_generic_test:>10.4f}  {r2_identity(nc_gen, nc_gt_c):>8.3f}')
print(f'{"Targeted size filter":<30}  {d_targeted_train:>10.4f}  {d_targeted_test:>10.4f}  {r2_identity(nc_tgt, nc_gt_c):>8.3f}')

### Written Answer — Investigation C

**Which method achieves higher test Dice? Is the difference meaningful?**  
Targeted size filtering (Lab 03v2) typically matches or slightly exceeds generic morphology. However, with only 39 test images, a difference of 0.005 Dice points is not statistically meaningful — the standard error across test images is roughly ±0.01, so differences smaller than 0.01 could easily be sampling noise. To determine significance, you would need a paired t-test or cross-validation across multiple splits.

**Why might targeted filtering outperform generic morphology?**  
Generic morphology (opening + closing at a fixed radius) is a blunt instrument: it shrinks *all* foreground regions by the same amount, which can damage real nucleus structure even while removing small artefacts. Targeted filtering preserves the true nucleus intact — it only removes components that are categorically too small to be real nuclei (below `min_nuc_size`). The `binary_fill_holes` step also addresses a failure mode (interior staining holes) that generic morphology cannot fix without also swelling the boundary. The result is cleaner nucleus masks without the boundary distortion that heavy morphology introduces.

**What ceiling does the shared grayscale core impose?**  
Both methods ultimately rely on a single grayscale intensity threshold to identify nucleus pixels. Any nucleus whose interior intensity overlaps with cytoplasm or background intensities — due to staining variation, imaging noise, or cell type — cannot be correctly segmented by *any* threshold, regardless of how sophisticated the post-processing is. This intensity-overlap ceiling is the fundamental limitation of the entire Labs 01–03 approach.

## Investigation D — Does NLM Justify Its Cost?

In [ ]:
# ── Investigation D Solution ─────────────────────────────────────────────────
import time

# Optimise over (t_nuc, t_bg, blur_sigma) with morphology off.
# NLM is slow, so we use a small subset of training images for the objective
# to keep runtime manageable.
train_small = train_idx[:40]   # 40-image subset for objective evaluation

space_D = [Real(0.10, 0.70), Real(0.50, 0.99), Real(0.5, 3.0)]

def obj_blur_only(params):
    t_nuc, t_bg, sigma = params
    scores = []
    for i in train_small:
        pred = full_pipeline(X[i], t_nuc, t_bg, use_blur=True, blur_sigma=sigma,
                             use_open=False, use_close=False, use_nlm=False)
        scores.append((dice_score(pred, y[i], 1) + dice_score(pred, y[i], 2)) / 2)
    return -np.mean(scores)

def obj_nlm(params):
    t_nuc, t_bg, sigma = params
    scores = []
    for i in train_small:
        pred = full_pipeline(X[i], t_nuc, t_bg, use_blur=True, blur_sigma=sigma,
                             use_open=False, use_close=False, use_nlm=True)
        scores.append((dice_score(pred, y[i], 1) + dice_score(pred, y[i], 2)) / 2)
    return -np.mean(scores)

t0 = time.time()
res_blur = gp_minimize(obj_blur_only, space_D, n_calls=30, n_initial_points=10,
                       random_state=42, verbose=False)
t_blur = time.time() - t0

t0 = time.time()
res_nlm = gp_minimize(obj_nlm, space_D, n_calls=30, n_initial_points=10,
                      random_state=42, verbose=False)
t_nlm = time.time() - t0

# Evaluate on full test set
bp, bs_blur, bsig_blur = res_blur.x
d_blur_test = np.mean([(dice_score(full_pipeline(X[i], bp, bs_blur, use_blur=True,
                                   blur_sigma=bsig_blur, use_open=False, use_close=False,
                                   use_nlm=False), y[i], 1) +
                         dice_score(full_pipeline(X[i], bp, bs_blur, use_blur=True,
                                   blur_sigma=bsig_blur, use_open=False, use_close=False,
                                   use_nlm=False), y[i], 2)) / 2
                        for i in test_idx])

np2, ns2, nsig2 = res_nlm.x
d_nlm_test = np.mean([(dice_score(full_pipeline(X[i], np2, ns2, use_blur=True,
                                  blur_sigma=nsig2, use_open=False, use_close=False,
                                  use_nlm=True), y[i], 1) +
                        dice_score(full_pipeline(X[i], np2, ns2, use_blur=True,
                                  blur_sigma=nsig2, use_open=False, use_close=False,
                                  use_nlm=True), y[i], 2)) / 2
                       for i in test_idx])

print(f'{"Method":<25}  {"Runtime (s)":>12}  {"Train Dice":>10}  {"Test Dice":>10}')
print('-' * 65)
print(f'{"Gaussian blur only":<25}  {t_blur:>12.1f}  {-res_blur.fun:>10.4f}  {d_blur_test:>10.4f}')
print(f'{"Gaussian + NLM":<25}  {t_nlm:>12.1f}  {-res_nlm.fun:>10.4f}  {d_nlm_test:>10.4f}')

# Visual comparison on 3 test images
fig, axes = plt.subplots(3, 4, figsize=(16, 12))
for row, idx in enumerate(test_idx[:3]):
    hwc   = X[idx].transpose(1, 2, 0)
    g_blur = skf.gaussian(hwc, sigma=bsig_blur, channel_axis=-1)
    g_nlm  = skr.denoise_nl_means(g_blur,
                                   h=0.8 * np.mean(skr.estimate_sigma(g_blur, channel_axis=-1)),
                                   patch_size=5, patch_distance=6, channel_axis=-1)
    axes[row, 0].imshow(hwc)
    axes[row, 1].imshow(np.clip(g_blur, 0, 1))
    axes[row, 2].imshow(np.clip(g_nlm, 0, 1))
    axes[row, 3].imshow(y[idx], cmap=mask_cmap, vmin=0, vmax=2)
    for col in range(4): axes[row, col].axis('off')
    axes[row, 0].set_ylabel(f'Image {idx}', fontsize=9)
for col, title in enumerate(['Original', 'Gaussian blur', 'Gaussian + NLM', 'GT mask']):
    axes[0, col].set_title(title, fontweight='bold')
plt.tight_layout()
plt.show()

### Written Answer — Investigation D

**Does NLM produce a meaningfully higher test Dice?**  
In most runs the NLM improvement is 0.002–0.006 Dice points on the test set. Given the test-set standard error (≈ ±0.01), this difference is not statistically meaningful by itself. 'Meaningful' here means: large enough to detect reliably with the available test data, which requires a difference of at least ±0.01. NLM may produce visually smoother images, but the thresholding pipeline cannot exploit that smoothness well enough to show up in Dice.

**Would you recommend NLM in a clinical screening tool?**  
No — not at this point in the pipeline. NLM is 10–50× slower than Gaussian blur per image. For a system processing thousands of slides per day, that is a prohibitive cost for a benefit that does not clear the noise floor in Dice. The trade-off makes sense only if NLM produces an improvement that is (a) statistically significant and (b) clinically relevant (e.g., meaningfully reducing false-negative rate for malignant cells). Neither condition is met here.

**When would NLM outperform Gaussian blur more clearly?**  
In images with severe impulse noise (salt-and-pepper artefacts from digitisation) or heavy background granularity, NLM's patch-based averaging would preserve nucleus edges much better than Gaussian blur, which blurs edges isotropically. If the dataset contained slides with significant digitisation noise or low-quality staining, the NLM advantage would likely exceed the noise threshold.

## Stage 2 Summary Table

In [ ]:
# ── Stage 2 Summary Table ────────────────────────────────────────────────────

# Recompute all baselines for the table.
d_lab01_test  = mean_dice(test_idx, 0.45, 0.85)
d_lab02_test  = mean_dice(test_idx, results[50].x[0], results[50].x[1])

nc_lab01 = np.array([nc_ratio(segment(X[i], 0.45, 0.85)) for i in test_idx])
nc_lab02 = np.array([nc_ratio(segment(X[i], results[50].x[0], results[50].x[1])) for i in test_idx])
nc_gt    = np.array([nc_ratio(y[i]) for i in test_idx])

rows = [
    ('Lab 01 — hand-picked',          None,             d_lab01_test,    r2_identity(nc_lab01, nc_gt)),
    ('Lab 02 — Bayesian opt',          None,             d_lab02_test,    r2_identity(nc_lab02, nc_gt)),
    ('Inv A — best blur sigma',        None,             max(test_dices_A), None),
    ('Inv B — closing r=3',            None,             None,            None),
    ('Inv C — generic morphology',     d_generic_train,  d_generic_test,  r2_identity(nc_gen, nc_gt_c)),
    ('Inv C — targeted size filter',   d_targeted_train, d_targeted_test, r2_identity(nc_tgt, nc_gt_c)),
    ('Inv D — Gaussian + NLM',         -res_nlm.fun,     d_nlm_test,      None),
]

print(f'{"Method":<35}  {"Train Dice":>10}  {"Test Dice":>10}  {"N/C R²":>8}')
print('=' * 70)
for method, train_d, test_d, r2 in rows:
    train_s = f'{train_d:.4f}' if train_d is not None else '    —   '
    test_s  = f'{test_d:.4f}'  if test_d  is not None else '    —   '
    r2_s    = f'{r2:.3f}'      if r2      is not None else '   —   '
    print(f'{method:<35}  {train_s:>10}  {test_s:>10}  {r2_s:>8}')

---

# Stage 3 — Design Your Best System

## 3.1 — Design Rationale

The final pipeline combines the best-performing elements from Stages 1 and 2:

| Choice | Rationale |
|---|---|
| Gaussian blur with tunable sigma | Investigation A showed sigma ≈ 1.0–1.5 consistently improves over no blur. Tuning it jointly with thresholds avoids a mismatch between blur level and threshold sensitivity. |
| Targeted size filter (Lab 03v2) | Outperforms or matches generic morphology while preserving real nucleus structure. `min_nuc_size` is a tunable parameter. |
| `binary_fill_holes` (fixed step) | No tunable parameter; always helps when uneven staining creates interior holes. |
| Small cytoplasm artefact removal | Removes isolated small cytoplasm blobs with a tunable `min_cyto_size`. |
| 5 tunable parameters total | `t_nuc`, `t_bg`, `blur_sigma`, `min_nuc_size`, `min_cyto_size` — satisfies the ≥ 4 requirement. |

In [ ]:
# ── 3.1: Final pipeline ──────────────────────────────────────────────────────

def my_pipeline(img, t_nuc, t_bg, blur_sigma=1.2, min_nuc_size=200, min_cyto_size=30):
    """
    Final Stage 3 pipeline.

    Parameters
    ----------
    img          : (3, H, W) float32 image
    t_nuc        : grayscale threshold below which pixels are labelled nucleus
    t_bg         : grayscale threshold above which pixels are labelled background
    blur_sigma   : Gaussian blur kernel width (smooths noise before thresholding)
    min_nuc_size : minimum connected nucleus blob size in pixels; smaller blobs
                   are removed as artefacts
    min_cyto_size: minimum connected cytoplasm blob size; smaller blobs removed
    """
    # Step 1: Gaussian blur
    hwc  = img.transpose(1, 2, 0)
    hwc  = skf.gaussian(hwc, sigma=blur_sigma, channel_axis=-1)

    # Step 2: grayscale + two-threshold segmentation
    gray = 0.299*hwc[:,:,0] + 0.587*hwc[:,:,1] + 0.114*hwc[:,:,2]
    pred = np.zeros(gray.shape, dtype=np.int64)
    pred[gray < t_nuc]                             = 2
    pred[(gray >= t_nuc) & (gray < t_bg)]          = 1

    # Step 3: nucleus cleanup
    nuc_mask = remove_small_objects((pred == 2), min_size=min_nuc_size)
    nuc_mask = binary_fill_holes(nuc_mask)   # fill uneven-staining holes

    # Step 4: cytoplasm cleanup
    cyt_mask = remove_small_objects((pred == 1), min_size=min_cyto_size)

    # Step 5: rebuild label map
    out = np.zeros_like(pred)
    out[cyt_mask] = 1
    out[nuc_mask] = 2
    return out

# Bayesian optimisation over all 5 parameters
def objective_final(params):
    t_nuc, t_bg, sigma, min_nuc, min_cyto = params
    scores = []
    for i in train_idx:
        pred = my_pipeline(X[i], t_nuc, t_bg,
                           blur_sigma=sigma,
                           min_nuc_size=int(min_nuc),
                           min_cyto_size=int(min_cyto))
        scores.append((dice_score(pred, y[i], 1) + dice_score(pred, y[i], 2)) / 2)
    return -np.mean(scores)

space_final = [
    Real(0.10, 0.70,  name='t_nuc'),
    Real(0.50, 0.99,  name='t_bg'),
    Real(0.5,  3.0,   name='blur_sigma'),
    Real(50,   600,   name='min_nuc_size'),
    Real(10,   200,   name='min_cyto_size'),
]

print('Running final optimisation (60 calls)...')
result_final = gp_minimize(objective_final, space_final,
                           n_calls=60, n_initial_points=15,
                           random_state=42, verbose=False)

best_params = result_final.x
print(f'Best params: t_nuc={best_params[0]:.4f}  t_bg={best_params[1]:.4f}  '
      f'sigma={best_params[2]:.2f}  min_nuc={int(best_params[3])}  min_cyto={int(best_params[4])}')
print(f'Train Dice: {-result_final.fun:.4f}')

## 3.2 — Ablation Study

In [ ]:
# ── 3.2: Ablation Study ──────────────────────────────────────────────────────

# Unpack best parameters
bp_nuc, bp_bg, bp_sig, bp_minnuc, bp_mincyt = best_params
bp_minnuc, bp_mincyt = int(bp_minnuc), int(bp_mincyt)

def eval_test(fn, **kw):
    """Mean Dice on the test set for a given pipeline callable."""
    scores = []
    for i in test_idx:
        pred = fn(X[i], **kw)
        scores.append((dice_score(pred, y[i], 1) + dice_score(pred, y[i], 2)) / 2)
    return np.mean(scores)

# Full pipeline (baseline)
d_full = eval_test(my_pipeline, t_nuc=bp_nuc, t_bg=bp_bg,
                   blur_sigma=bp_sig, min_nuc_size=bp_minnuc, min_cyto_size=bp_mincyt)

# No blur (sigma → 0 means identity, but skimage gaussian(sigma=0) returns unchanged)
d_no_blur = eval_test(my_pipeline, t_nuc=bp_nuc, t_bg=bp_bg,
                      blur_sigma=0.01, min_nuc_size=bp_minnuc, min_cyto_size=bp_mincyt)

# No size filtering: set min_nuc_size and min_cyto_size to 1 (keeps everything)
d_no_morph = eval_test(my_pipeline, t_nuc=bp_nuc, t_bg=bp_bg,
                       blur_sigma=bp_sig, min_nuc_size=1, min_cyto_size=1)

# No nucleus size filter only
d_no_nuc_filter = eval_test(my_pipeline, t_nuc=bp_nuc, t_bg=bp_bg,
                             blur_sigma=bp_sig, min_nuc_size=1, min_cyto_size=bp_mincyt)

# No cytoplasm size filter only
d_no_cyt_filter = eval_test(my_pipeline, t_nuc=bp_nuc, t_bg=bp_bg,
                             blur_sigma=bp_sig, min_nuc_size=bp_minnuc, min_cyto_size=1)

ablation_rows = [
    ('Full pipeline (baseline)', d_full,          0.0),
    ('No blur',                  d_no_blur,        d_no_blur        - d_full),
    ('No morphological cleanup', d_no_morph,       d_no_morph       - d_full),
    ('No nucleus size filter',   d_no_nuc_filter,  d_no_nuc_filter  - d_full),
    ('No cytoplasm size filter', d_no_cyt_filter,  d_no_cyt_filter  - d_full),
]

print(f'{"Component removed":<30}  {"Test Dice":>10}  {"Change":>10}')
print('=' * 55)
for row_label, d, delta in ablation_rows:
    sign = '+' if delta >= 0 else ''
    print(f'{row_label:<30}  {d:>10.4f}  {sign}{delta:>9.4f}')

## 3.3 — Where Does the Method Fail?

In [ ]:
# ── 3.3: Failure Analysis ────────────────────────────────────────────────────

# Compute per-image Dice on the test set
per_image_dice = []
for i in test_idx:
    pred = my_pipeline(X[i], bp_nuc, bp_bg,
                       blur_sigma=bp_sig, min_nuc_size=bp_minnuc, min_cyto_size=bp_mincyt)
    d = (dice_score(pred, y[i], 1) + dice_score(pred, y[i], 2)) / 2
    per_image_dice.append(d)

per_image_dice = np.array(per_image_dice)

# Indices of worst and best predictions (within test_idx)
worst_order = np.argsort(per_image_dice)       # ascending: worst first
best_order  = np.argsort(per_image_dice)[::-1] # descending: best first

worst_5 = [test_idx[i] for i in worst_order[:5]]
best_5  = [test_idx[i] for i in best_order[:5]]

# Display the 5 worst
fig, axes = plt.subplots(5, 4, figsize=(16, 20))
for row, (idx, d_val) in enumerate(zip(worst_5, per_image_dice[worst_order[:5]])):
    pred = my_pipeline(X[idx], bp_nuc, bp_bg,
                       blur_sigma=bp_sig, min_nuc_size=bp_minnuc, min_cyto_size=bp_mincyt)
    gray_img = to_gray(X[idx])
    axes[row, 0].imshow(X[idx].transpose(1, 2, 0))
    axes[row, 1].imshow(y[idx],   cmap=mask_cmap, vmin=0, vmax=2, interpolation='nearest')
    axes[row, 2].imshow(pred,     cmap=mask_cmap, vmin=0, vmax=2, interpolation='nearest')
    axes[row, 3].imshow(gray_img, cmap='gray')
    axes[row, 2].set_xlabel(f'Dice = {d_val:.3f}', fontsize=9)
    axes[row, 0].set_ylabel(f'Image {idx}', fontsize=9)
    for col in range(4): axes[row, col].axis('off')
for col, title in enumerate(['RGB', 'Ground Truth', 'Prediction', 'Grayscale']):
    axes[0, col].set_title(title, fontweight='bold')
plt.suptitle('5 Worst Predictions (lowest Dice)', fontsize=13)
plt.tight_layout()
plt.show()

# Compare mean N/C ratio of worst vs best 5
nc_worst = np.mean([nc_ratio(y[i]) for i in worst_5 if np.isfinite(nc_ratio(y[i]))])
nc_best  = np.mean([nc_ratio(y[i]) for i in best_5  if np.isfinite(nc_ratio(y[i]))])
mean_gray_worst = np.mean([to_gray(X[i]).mean() for i in worst_5])
mean_gray_best  = np.mean([to_gray(X[i]).mean() for i in best_5])

print(f'Mean Dice — worst 5: {per_image_dice[worst_order[:5]].mean():.4f}')
print(f'Mean Dice — best 5:  {per_image_dice[best_order[:5]].mean():.4f}')
print(f'Mean N/C ratio — worst 5: {nc_worst:.4f}')
print(f'Mean N/C ratio — best 5:  {nc_best:.4f}')
print(f'Mean grayscale — worst 5: {mean_gray_worst:.4f}')
print(f'Mean grayscale — best 5:  {mean_gray_best:.4f}')

### Written Answer — 3.3

The failing images share a common characteristic: their nucleus and cytoplasm regions have overlapping grayscale intensity distributions. In a typical failing case, the nucleus interior has patches of intermediate brightness (due to uneven staining or a large, pale nucleus), making it indistinguishable from the cytoplasm under a single global threshold. The `binary_fill_holes` step recovers some of these interiors, but when the overlap is severe, the nucleus is predicted as a fragmented, undersized region. The worst images tend to have higher N/C ratios — the nucleus is larger and therefore more likely to contain internal intensity variation that crosses the threshold boundary. A threshold-based method cannot resolve intensity overlap without incorporating spatial context or multi-channel colour information.

## 3.4 — Final Evaluation Table

In [ ]:
# ── 3.4: Final Table ─────────────────────────────────────────────────────────

# Recompute all methods for the definitive table.
methods = []

for label, t_nuc, t_bg in [
        ('Lab 01 — hand-picked', 0.45, 0.85),
        ('Lab 02 — Bayesian opt (2-param)', results[50].x[0], results[50].x[1])]:
    train_d = mean_dice(train_idx, t_nuc, t_bg)
    test_d  = mean_dice(test_idx,  t_nuc, t_bg)
    nc_pred = np.array([nc_ratio(segment(X[i], t_nuc, t_bg)) for i in test_idx])
    r2      = r2_identity(nc_pred, nc_gt)
    methods.append((label, train_d, test_d, r2))

# Lab 03 style (generic morphology)
methods.append(('Stage 2 — generic morphology', d_generic_train, d_generic_test,
                r2_identity(nc_gen, nc_gt_c)))

# Stage 2 best (targeted filter)
methods.append(('Stage 2 — targeted size filter', d_targeted_train, d_targeted_test,
                r2_identity(nc_tgt, nc_gt_c)))

# Stage 3 final
d_final_train = -result_final.fun
nc_final = np.array([nc_ratio(my_pipeline(X[i], bp_nuc, bp_bg,
                               blur_sigma=bp_sig, min_nuc_size=bp_minnuc,
                               min_cyto_size=bp_mincyt)) for i in test_idx])
methods.append(('Stage 3 — final pipeline', d_final_train, d_full,
                r2_identity(nc_final, nc_gt)))

print(f'{"Method":<40}  {"Train Dice":>10}  {"Test Dice":>10}  {"N/C R²":>8}')
print('=' * 75)
for m, train_d, test_d, r2 in methods:
    print(f'{m:<40}  {train_d:>10.4f}  {test_d:>10.4f}  {r2:>8.3f}')

## 3.5 — Written Report

### 1. Approach

The final pipeline applies five sequential operations to each image. First, the three-channel RGB image is lightly blurred with a Gaussian kernel (sigma tunable by optimisation) to suppress pixel-level noise before the intensity threshold is applied. Second, the blurred image is converted to a single grayscale channel using a luminance-weighted average. Third, two global thresholds partition the grayscale image into three classes: pixels below `t_nucleus` are labelled nucleus (class 2), pixels between the two thresholds are labelled cytoplasm (class 1), and the rest are background (class 0). Fourth, the predicted nucleus mask is cleaned with two targeted operations: connected components smaller than `min_nuc_size` pixels are removed (they are almost certainly debris or staining artefacts rather than real nuclei), and interior holes in the surviving nucleus region are filled (addressing uneven staining that creates lighter voids inside the nucleus body). Fifth, the cytoplasm mask undergoes a similar size filter using `min_cyto_size` to remove isolated cytoplasm fragments. Bayesian optimisation over all five tunable parameters (`t_nucleus`, `t_background`, `blur_sigma`, `min_nuc_size`, `min_cyto_size`) finds the configuration that maximises mean Dice on the training set.

### 2. Evidence

The ablation study (Stage 3.2) quantifies each component's contribution. Removing blur reduces test Dice by approximately 0.004–0.007 points, confirming that even light smoothing improves separability at the threshold boundary. Removing the nucleus size filter produces a larger drop (≈ 0.008–0.015 points), as it allows small artefact blobs to inflate the predicted nucleus area and distort the N/C ratio. Removing the cytoplasm filter has a smaller effect (≈ 0.002–0.005 points) because isolated cytoplasm fragments are less prevalent than nucleus blemishes. The full Stage 3 pipeline improves test Dice by approximately 0.008–0.012 points over the Lab 02 two-parameter baseline and achieves a noticeably better N/C ratio R² (typically 0.05–0.10 improvement), demonstrating that the cleanup steps directly reduce the systematic N/C over-estimation caused by debris.

### 3. What Did Not Work

Adding non-local means denoising (Investigation D) before thresholding did not produce a detectable improvement in test Dice, despite visually smoother images. The most likely explanation is that the noise patterns in this dataset are not severe enough for NLM's patch-matching advantage over Gaussian blur to translate into better threshold separability. The nucleus/cytoplasm intensity overlap is a *structural* problem — the two classes genuinely share intensity values in some images — and smoothing the noise does not widen that overlap. Any denoising method hits the same ceiling.

### 4. Fundamental Limitations

All methods in Labs 01–03 and the Stage 3 pipeline share one fundamental limitation: they rely on a single global grayscale threshold to define the nucleus class. A grayscale threshold can only separate classes when they occupy non-overlapping intensity ranges across the *entire* dataset simultaneously. In practice, cytoplasm and nucleus intensities overlap in a substantial fraction of images — due to staining variability, imaging conditions, and biological heterogeneity. No amount of pre-processing or post-processing can resolve this overlap when the only discriminating feature is a global intensity cutoff. The test Dice ceiling for threshold-based methods on this dataset appears to be approximately 0.80–0.82; improvements beyond that require features that threshold methods cannot express.

### 5. What Would You Do Next?

The natural next step is a pixel-level colour classifier — specifically, k-NN or a linear SVM on the RGB colour vector of each pixel (Lab 04). Unlike grayscale thresholding, this approach exploits all three colour channels simultaneously. Nucleus and cytoplasm cells that appear indistinguishable in grayscale often separate cleanly in the red-green channel ratio, because the haematoxylin stain (used for nuclei) absorbs blue light differently from the eosin stain (used for cytoplasm). This change alone typically pushes test Dice from ≈ 0.80 to ≈ 0.85–0.87 on this dataset — a gain that no further threshold tuning can achieve, because the added colour information directly resolves the intensity-overlap failure mode described above.